# Notebook 1 — Explore the HARD-Lite data and build features

This notebook downloads the public HARD-Lite ransomware dataset, explores it and builds the feature table used in Notebook 2.

**Data source:** Woralert, Liu and Blasingame, *HARD-Lite: A Lightweight Hardware Anomaly Realtime Detection Framework Targeting Ransomware*, IEEE Transactions on Circuits and Systems I, 2023. Repository: <https://github.com/chutitepw/HPC_Ransomware_Detection>. Six hardware performance counters were recorded for the whole system every 10 ms, once during normal operation (benign) and once while each of 12 ransomware families was running. Each recording was made twice: one run is in `train/`, a separate run is in `test/`.

> **Safety note:** only recorded numbers are used. No malware is downloaded or run.

## Step 1 — Download

The files come from a fixed version (commit) of the authors' repository. Files already in `data/` are not downloaded again.

In [ ]:
import os, re, glob, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "data"
REPO_RAW = ("https://raw.githubusercontent.com/chutitepw/HPC_Ransomware_Detection/"
            "bc08004f5371cbe05e12bad7143bbfe36a2ef7dc/data")

FAMILIES = ["alphv", "blackcat", "c", "cryptsky", "golang", "hellokitty", "holycrypt",
            "monti", "ransomexx", "revil", "sodinokibi", "tellyouthepass"]
FILES = {
    "train": ["benign-1.csv"] + [f"ransom-{f}-1.csv" for f in FAMILIES],
    "test":  ["benign-51.csv"]
             + [f"ransom-{f}-51.csv" for f in FAMILIES if f != "tellyouthepass"]
             + ["ransom-tellyouthepass-6.csv"],
}

for split, names in FILES.items():
    os.makedirs(os.path.join(DATA_DIR, split), exist_ok=True)
    for name in names:
        path = os.path.join(DATA_DIR, split, name)
        if not os.path.exists(path):
            print("Downloading", split, name)
            urllib.request.urlretrieve(f"{REPO_RAW}/{split}/{name}", path)
print("Data ready:", sum(len(v) for v in FILES.values()), "files")

## Step 2 — Load

Each CSV has one row per 10 ms and six counter columns named after AMD event codes. They are renamed:

| Column in file | New name | What it counts |
|---|---|---|
| `c2` | `branches` | branch instructions retired |
| `c0` | `instructions` | instructions retired |
| `729` | `dcache_accesses` | data-cache accesses |
| `129` | `loads` | memory load operations |
| `229` | `stores` | memory store operations |
| `ff9a` | `llc_misses` | last-level (L3) cache misses |

Added columns: `label` (0 = benign, 1 = ransomware), `family`, `run` (which recording) and `t_sec` (seconds since the start of the recording). Rows with zero instructions (the start of most files) are dropped.

In [ ]:
COUNTER_NAMES = {"c2": "branches", "c0": "instructions", "729": "dcache_accesses",
                 "129": "loads", "229": "stores", "ff9a": "llc_misses"}
COUNTERS = list(COUNTER_NAMES.values())

def load_split(split):
    frames = []
    for path in sorted(glob.glob(os.path.join(DATA_DIR, split, "*.csv"))):
        run = os.path.basename(path)[:-4]
        m = re.match(r"(benign|ransom-(.+))-(\d+)$", run)
        family = "benign" if m.group(1) == "benign" else m.group(2)
        df = pd.read_csv(path).rename(columns=COUNTER_NAMES)
        df = df[df["instructions"] > 0].copy()
        df["label"] = (df["type"] == "ransom").astype(int)
        df["family"] = family
        df["run"] = run
        df["t_sec"] = np.arange(len(df)) * 0.01
        frames.append(df.drop(columns="type"))
    return pd.concat(frames, ignore_index=True)

rows_train = load_split("train")
rows_test = load_split("test")
print("train rows:", len(rows_train), " test rows:", len(rows_test))
rows_train.head()

## Step 3 — Overview of the recordings

In [ ]:
def overview(rows):
    return (rows.groupby(["run", "family", "label"])
                .agg(rows_10ms=("instructions", "size"),
                     median_instructions=("instructions", "median"))
                .assign(seconds=lambda d: d["rows_10ms"] / 100)
                .reset_index()
                .sort_values(["label", "family"]))

pd.set_option("display.float_format", "{:,.0f}".format)
display(overview(rows_train))
display(overview(rows_test))
pd.reset_option("display.float_format")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for run in ["benign-1", "ransom-revil-1", "ransom-monti-1", "ransom-cryptsky-1"]:
    r = rows_train[(rows_train["run"] == run) & (rows_train["t_sec"] < 60)]
    ax.plot(r["t_sec"], r["instructions"], label=run, linewidth=0.7)
ax.set_yscale("log")
ax.set_xlabel("time (s)")
ax.set_ylabel("instructions per 10 ms (log scale)")
ax.set_title("First 60 seconds of a few recordings")
ax.legend()
plt.show()

*Your observations about the recordings:*

## Task 1.1 — Feature design

Implement `build_examples(rows)`. It must return a DataFrame with the columns `label`, `family` and `run`, plus the numeric feature columns of your choice. Notebook 2 treats **every other column as a model input**.

You decide what a single example is and which features describe it. Justify each decision in the markdown cell below the code.

In [ ]:
def build_examples(rows):
    # Task 1.1
    raise NotImplementedError


feat_train = build_examples(rows_train)
feat_test = build_examples(rows_test)
print("train examples:", feat_train.shape, " test examples:", feat_test.shape)
print("train class counts:", feat_train["label"].value_counts().to_dict())
feat_train.head()

*Your justification:*

## Task 1.2 — Evidence

Produce the tables and figures that support, or challenge, your feature design. Describe what they show in the markdown cell below.

In [ ]:
# Task 1.2

*Your observations:*

## Step 4 — Save for Notebook 2

In [ ]:
feat_train.to_csv("features_train.csv", index=False)
feat_test.to_csv("features_test.csv", index=False)
print("Saved features_train.csv and features_test.csv")